In [2]:
%pip install pandas BeautifulSoup4

Looking in indexes: https://mirror-pypi.runflare.com/simple
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.9 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.9 MB 941.6 kB/s eta 0:00:10
   -- ------------------------------------- 0.5/9.9 MB 941.6 kB/s eta 0:00:10
   --- ------------------------------------ 0.8/9.9 MB 851.7 kB/s eta 0:00:11
   ---- ----------------------------------- 1.0/9.9 MB 818.1 kB/s eta 0:00:11
   ---- ----------------------------------- 1.0/9.9 MB 818.1 kB/s eta 0:00:11
   ---- ----------------------------------- 1.0/9.9 MB 818.1 kB/s eta 0:00:11
   ----- ---------------------------------- 1.3/9.9 MB 702.9 kB/s eta 0:00:13
   ----- ---------------------------------- 1.3/9.9 MB 702.9 kB/s eta 0:00:13
   ------ --------------------------------- 1.6/9.9 MB 675.6 kB/s eta 0:00:13
   ------ ---------------------

In [ ]:
from pprint import pp
import json
import pandas as pd 
import sys
import re
import requests
from bs4 import BeautifulSoup


# url = "https://www.downloadha.com/tag/%D8%A8%D8%A7%D8%B2%DB%8C-aaa/page/28/"
# res = requests.get(url)

def clean_ctn(text):
    visits = re.findall(r"[\d,]+", text)[0]
    visits_int = int(visits.replace(",", ""))
    return visits_int
    
PGN = 50
df = pd.DataFrame()

for i in range(1, PGN):
    url = f"https://www.downloadha.com/tag/%D8%A8%D8%A7%D8%B2%DB%8C-aaa/page/{i}/"
    res = requests.get(url)
    soup = BeautifulSoup(res.text,"html.parser")

    data = {
        "titles" : [i.text for i in soup.select('.post h1')],
        "catgory" : [i.text for i in soup.select('.breadcrumbs, .post-categories')],
        "publish" : [i.text for i in soup.select('.post-meta .post-date time.published')],
        "author" : [i.text for i in soup.select('.post-meta .byline .author a')],
        "ctn" : [clean_ctn(i.text) for i in soup.select('.post-meta .post-view:nth-of-type(2)')],
        "comment" : [i.text for i in soup.select('.post-meta a[href$="#respond"]')],
        "img" : [i.attrs["src"] for i in soup.select('.post img')],
        "link" : [i.attrs["href"] for i in soup.select('.btn.btn-raised.btn-info')],
        "pageNum" : i,
    }
    
    df2 = pd.DataFrame(data)
    df = pd.concat([df, df2], ignore_index=True)

df = df.sort_values(by="ctn", ascending=False)
df.to_csv("game.csv")